# 01_sbatch_embeddings_runs

Firts onet embeddings comparing two large (BGE and E5) transformers models for onet and then for sample1000 targets.

In [11]:
# Wipe the Adzuna embeddings
!rm -rf /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/e5_large/*

# Wipe the ONET chunks and final files
!rm -f /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_onet/tmp_chunk_onet_*
!rm -f /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_onet/*.npy

# Wipe the old cosine similarity parts
!rm -rf /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/cosines/e5_large/*

## Onet embeddings

creates Onet roles and task/skills embeddings for onet jobs for bge and e5 model and quick evaluates output

In [12]:
import subprocess
import os
from datetime import datetime
import time
sbatch_dir = "/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_2_Embeddings_and_Cosine_sim/dev/sbatch/"
SBATCH_FILE = sbatch_dir + "embed_onet_large.sbatch"

# 1. Submit job with timestamp
start_submit_time = datetime.now()
print("Submitting job at:", start_submit_time.strftime("%Y-%m-%d %H:%M:%S"))

result = subprocess.run(
    ["sbatch", SBATCH_FILE],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError("sbatch submission failed")

job_id = result.stdout.strip().split()[-1]
print("Job ID:", job_id)

# 2. Check queue status
print("\nCurrent queue status:")
os.system(f"squeue -j {job_id}")

# 3. Wait until job finishes
print("\nWaiting for job to finish...")
while True:
    check = subprocess.run(
        ["squeue", "-j", job_id],
        capture_output=True,
        text=True
    )
    if job_id not in check.stdout:
        break
    time.sleep(20)

end_time = datetime.now()

# 4. Print timing info
print("\n===================================")
print("JOB SUBMITTED:", start_submit_time.strftime("%Y-%m-%d %H:%M:%S"))
print("JOB FINISHED:", end_time.strftime("%Y-%m-%d %H:%M:%S"))
print("TOTAL WALL TIME:", end_time - start_submit_time)
print("===================================\n")



Submitting job at: 2026-02-01 00:20:29
Submitted batch job 2094626

Job ID: 2094626

Current queue status:
  JOBID         USER PARTITION                     NAME ST TIME_LIMIT       TIME  TIME_LEFT NODES NODELIST(REASON)
2094626 autonomyluiz     workq   emb_onet_large_bge_gte PD    2:00:00       0:00    2:00:00     1 (None)

Waiting for job to finish...

JOB SUBMITTED: 2026-02-01 00:20:29
JOB FINISHED: 2026-02-01 00:21:29
TOTAL WALL TIME: 0:01:00.472213



# Now run target embeddings 

In [13]:
import subprocess, time, glob
from datetime import datetime
from pathlib import Path

SBATCH = "/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_2_Embeddings_and_Cosine_sim/dev/sbatch/embed_target_large.sbatch"
LOG_GLOB = "/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_2_Embeddings_and_Cosine_sim/dev/sbatch/logs/*.out"

print("submit:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
r = subprocess.run(["sbatch", SBATCH], capture_output=True, text=True)
print("stdout:", r.stdout.strip() or "<empty>")
print("stderr:", r.stderr.strip() or "<empty>")
r.check_returncode()

job_id = r.stdout.strip().split()[-1]
print("job_id:", job_id)

# wait until job appears
for _ in range(30):
    q = subprocess.run(["squeue", "-j", job_id], capture_output=True, text=True)
    if job_id in q.stdout:
        break
    time.sleep(2)

print("queue:\n", q.stdout.strip() or "<not in queue>")

# follow logs (best-effort)
print("\nlog tail (Ctrl+C to stop):")
try:
    while True:
        outs = sorted(glob.glob(LOG_GLOB))
        job_outs = [p for p in outs if job_id in Path(p).name]
        if job_outs:
            p = job_outs[-1]
            tail = subprocess.run(["tail", "-n", "40", p], capture_output=True, text=True).stdout
            print("\n---", Path(p).name, datetime.now().strftime("%H:%M:%S"), "---\n", tail.rstrip())
        else:
            print("waiting for log file...")
        q = subprocess.run(["squeue", "-j", job_id], capture_output=True, text=True)
        if job_id not in q.stdout:
            print("\njob finished:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            break
        time.sleep(10)
except KeyboardInterrupt:
    print("\nstopped tailing logs")


submit: 2026-02-01 00:21:29
stdout: Submitted batch job 2094627
stderr: <empty>
job_id: 2094627
queue:
 JOBID         USER PARTITION                     NAME ST TIME_LIMIT       TIME  TIME_LEFT NODES NODELIST(REASON)
2094627 autonomyluiz     workq        emb_bge_gte_small PD    6:00:00       0:00    6:00:00     1 (None)

log tail (Ctrl+C to stop):
waiting for log file...

--- emb_bge_gte_small_2094627_9.out 00:21:39 ---
 start: 2026-02-01T00:21:31+00:00
host:  nid010917
TASK_ID: 9
MONTH: adzuna_month05
SHARD: 1 of 2
INPUT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llama_summary_1000s/adzuna_month05.npz exists= True
BGE_SCRIPT: /projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_2_Embeddings_and_Cosine_sim/dev/sbatch/embed_target_large_bge.py exists= True
GTE_SCRIPT: /projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_2_Embeddings_and_Cosine_sim/dev/sbatch/embed_target_large_gte.py exists= True
cuda: 

In [14]:
import numpy as np
import hashlib
from pathlib import Path

# ---- CONFIG ----
month = "adzuna_month05"
shard = "01"   # "00" or "01"

bge_path = Path(
    f"/projects/a5u/adu_dev/aisi-economy-index/"
    f"aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/"
    f"embeddings/run_embed_target/bge_large/{month}/{month}_shard{shard}_of02.npz"
)

e5_path = Path(
    f"/projects/a5u/adu_dev/aisi-economy-index/"
    f"aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/"
    f"embeddings/run_embed_target/e5_large/{month}/{month}_shard{shard}_of02.npz"
)

print("BGE exists:", bge_path.exists(), bge_path)
print("E5  exists:", e5_path.exists(),  e5_path)
assert bge_path.exists() and e5_path.exists()

# ---- BYTE-LEVEL CHECK (definitive) ----
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

hb = sha256(bge_path)
he = sha256(e5_path)

print("\nSHA256:")
print("BGE:", hb)
print("E5 :", he)
print("same_bytes:", hb == he)

# ---- ARRAY-LEVEL CHECK (sanity) ----
b = np.load(bge_path, allow_pickle=True)
e = np.load(e5_path, allow_pickle=True)

for k in ["role_embeds", "taskskill_embeds"]:
    print(f"\n{k}:")
    print("  identical:", np.array_equal(b[k], e[k]))
    print("  max_abs_diff:", float(np.max(np.abs(b[k] - e[k]))))

print("\njob_ids identical:", np.array_equal(b["job_ids"], e["job_ids"]))


BGE exists: True /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/bge_large/adzuna_month05/adzuna_month05_shard01_of02.npz
E5  exists: False /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/e5_large/adzuna_month05/adzuna_month05_shard01_of02.npz


AssertionError: 

In [10]:
import numpy as np

E5_NPZ = (
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/"
    "embeddings/run_embed_target/e5_large/adzuna_month07/"
    "adzuna_month07_shard01_of02.npz"
)

z = np.load(E5_NPZ, allow_pickle=True)
X = z["role_embeds"].astype(np.float32)

# pairwise cosine stats
rng = np.random.default_rng(0)
n = X.shape[0]
i = rng.integers(0, n, size=20000)
j = rng.integers(0, n, size=20000)
j[i == j] = (j[i == j] + 1) % n

cos = (X[i] * X[j]).sum(axis=1)

print("E5 role pairwise cos:")
print(" mean:", float(cos.mean()))
print(" std :", float(cos.std()))
print(" p01 :", float(np.quantile(cos, 0.01)))
print(" p50 :", float(np.quantile(cos, 0.50)))
print(" p99 :", float(np.quantile(cos, 0.99)))


E5 role pairwise cos:
 mean: 0.7888750433921814
 std : 0.027368078008294106
 p01 : 0.7335683703422546
 p50 : 0.7868692278862
 p99 : 0.8656306266784668


In [15]:
import numpy as np
p = "/projects/.../run_embed_target/e5_large/adzuna_month07/adzuna_month07_shard01_of02.npz"
z = np.load(p, allow_pickle=True)
print("keys:", z.files)
print("shape:", z["role_embeds"].shape)
print("model_name:", z["model_name"] if "model_name" in z.files else "<missing>")


FileNotFoundError: [Errno 2] No such file or directory: '/projects/.../run_embed_target/e5_large/adzuna_month07/adzuna_month07_shard01_of02.npz'

In [16]:
import numpy as np

p = (
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/"
    "embeddings/run_embed_target/gte_large/"
    "adzuna_month05/adzuna_month05_shard01_of02.npz"
)

z = np.load(p, allow_pickle=True)
print("model_name:", z["model_name"])
print("shape:", z["role_embeds"].shape, z["taskskill_embeds"].shape)


FileNotFoundError: [Errno 2] No such file or directory: '/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/gte_large/adzuna_month05/adzuna_month05_shard01_of02.npz'

In [17]:
!ls -lh /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/*/adzuna_month05/adzuna_month05_shard01_of02.npz


-rw-r--r-- 1 autonomyluiz.a5u autonomyluiz.a5u 2,0M fev  1 00:21 /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/run_embed_target/bge_large/adzuna_month05/adzuna_month05_shard01_of02.npz


In [18]:
import numpy as np

p = (
  "/projects/a5u/adu_dev/aisi-economy-index/"
  "aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/"
  "embeddings/run_embed_target/bge_large/"
  "adzuna_month05/adzuna_month05_shard01_of02.npz"
)

z = np.load(p, allow_pickle=True)
print("keys:", z.files)
if "model_name" in z.files:
    print("model_name:", z["model_name"])


keys: ['job_ids', 'role_embeds', 'taskskill_embeds', 'month', 'shard_id', 'n_shards']


In [ ]:
import time
from pathlib import Path

LOG_DIR = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "nbs/AISI_demo/Stage_2_Embeddings_and_Cosine_sim/dev/sbatch/logs"
)

PATTERN = "gte_onet_and_adzuna"  # parte do nome do job

def find_latest_logs():
    outs = sorted(LOG_DIR.glob(f"*{PATTERN}*.out"), key=lambda p: p.stat().st_mtime)
    errs = sorted(LOG_DIR.glob(f"*{PATTERN}*.err"), key=lambda p: p.stat().st_mtime)
    return outs[-1] if outs else None, errs[-1] if errs else None

out_log, err_log = find_latest_logs()

print("OUT:", out_log)
print("ERR:", err_log)

def tail(path, n=20):
    if not path or not path.exists():
        return ["<no file>"]
    return path.read_text(errors="ignore").splitlines()[-n:]

while True:
    print("\n" + "=" * 80)
    print("STDOUT (last 20 lines)")
    print("\n".join(tail(out_log)))
    print("\nSTDERR (last 20 lines)")
    print("\n".join(tail(err_log)))
    time.sleep(15)
